# Game 1 - Multivariate Pair Trading (Coffee, G17)


## 1. Imports


In [ ]:
import os

try:
    from google.colab import files
    print("Google Colab detecte — upload le fichier coffee_dataset_G17.csv")
    uploaded = files.upload()
    CSV_PATH = list(uploaded.keys())[0]
except ImportError:
    CSV_PATH = 'coffee_dataset_G17.csv'
    print(f"Execution locale — fichier: {CSV_PATH}")

os.makedirs('figures', exist_ok=True)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr, zscore as _zscore
from statsmodels.tsa.stattools import adfuller
import gurobipy as gp
from gurobipy import GRB

pd.set_option('display.float_format', '{:.4f}'.format)
print('imports OK')

In [ ]:
CSV_PATH = 'coffee_dataset_G17.csv'

df_raw = pd.read_csv(CSV_PATH, index_col=0, parse_dates=True)
df_raw.index.name = 'Date'
df_raw = df_raw.sort_index()

print(f'Dimensions: {df_raw.shape}')
print(f'Periode: {df_raw.index[0].date()} -> {df_raw.index[-1].date()}')
df_raw.head()

## 2. Nettoyage


In [ ]:
df_clean = df_raw.dropna(thresh=int(0.80 * len(df_raw)), axis=1)
df_clean = df_clean.ffill().dropna()

for col in df_clean.columns:
    zs = _zscore(df_clean[col].pct_change().dropna())
    bad_idx = df_clean.index[1:][np.abs(zs) > 5]
    df_clean.loc[bad_idx, col] = np.nan
df_clean = df_clean.ffill().dropna()

ANCHOR = 'KC=F'
PROXIES = [c for c in df_clean.columns if c in ['DBA']]
STOCKS = [c for c in df_clean.columns if c not in [ANCHOR] + PROXIES]

print(f'Anchor: {ANCHOR}')
print(f'Proxies: {PROXIES}')
print(f'Stocks: {STOCKS}')
print(f'N obs: {len(df_clean)}')

## 3. Test de cointegration


In [ ]:
def engle_granger_test(y, x, significance=0.10):
    common = y.index.intersection(x.index)
    y, x = y.loc[common], x.loc[common]
    if len(y) < 30:
        return 1.0, False, 0.0
    X = np.column_stack([np.ones(len(x)), x.values])
    beta = np.linalg.lstsq(X, y.values, rcond=None)[0]
    resid = y.values - X @ beta
    _, pvalue, *_ = adfuller(resid, regression='c')
    return pvalue, pvalue < significance, float(beta[1])


log_prices = np.log(df_clean)
anchor_log = log_prices[ANCHOR]

results = []
_hedge_cache = {}
for stock in STOCKS + PROXIES:
    stock_log = log_prices[stock]
    rets_a = anchor_log.diff().dropna()
    rets_s = stock_log.diff().dropna()
    common = rets_a.index.intersection(rets_s.index)
    corr, _ = pearsonr(rets_a.loc[common], rets_s.loc[common])
    pval, is_coint, hedge = engle_granger_test(stock_log, anchor_log)
    _hedge_cache[stock] = hedge
    results.append({
        'Asset': stock,
        'Correlation': round(corr, 4),
        'EG p-value': round(pval, 4),
        'Cointegrated': is_coint,
        'Hedge Ratio': round(hedge, 4)
    })

coint_df = pd.DataFrame(results).sort_values('EG p-value')
coint_df

In [ ]:
N_SELECT = 5
selected = coint_df.sort_values('EG p-value').head(N_SELECT)['Asset'].tolist()
print(f'Basket selectionne: {selected}')

hedge_ratios = {s: _hedge_cache[s] for s in selected}
for s, h in hedge_ratios.items():
    print(f'  {s}: beta = {h:.4f}')

## 4. Optimisation Gurobi


In [ ]:
def optimize_basket(selected_stocks, log_prices, hedge_ratios, allow_short=True, verbose=False):
    n = len(selected_stocks)
    rets = log_prices[selected_stocks].diff().dropna()
    Sigma = rets.cov().values
    betas = np.array([hedge_ratios[s] for s in selected_stocks])

    model = gp.Model('MarketNeutralBasket')
    model.Params.OutputFlag = int(verbose)
    model.Params.TimeLimit = 60

    w = model.addVars(n, lb=-1.0 if allow_short else 0.0, ub=1.0, name='w')

    variance = gp.quicksum(Sigma[i, j] * w[i] * w[j] for i in range(n) for j in range(n))
    model.setObjective(variance, GRB.MINIMIZE)

    model.addConstr(gp.quicksum(w[i] for i in range(n)) == 1, 'fully_invested')
    model.addConstr(gp.quicksum(betas[i] * w[i] for i in range(n)) == 0, 'market_neutral')

    model.optimize()

    if model.status == GRB.OPTIMAL:
        return pd.Series([w[i].X for i in range(n)], index=selected_stocks)
    else:
        raise RuntimeError(f'Gurobi status: {model.status}')


FORMATION_END = '2022-12-31'
TRADING_START = '2023-01-01'

log_formation = log_prices.loc[:FORMATION_END]
log_trading = log_prices.loc[TRADING_START:]
prices_trading = df_clean.loc[TRADING_START:]

print(f'Formation: {log_formation.index[0].date()} -> {log_formation.index[-1].date()} ({len(log_formation)} j)')
print(f'Trading: {log_trading.index[0].date()} -> {log_trading.index[-1].date()} ({len(log_trading)} j)')

In [ ]:
hedge_formation = {}
for stock in selected:
    _, _, h = engle_granger_test(log_formation[stock], log_formation[ANCHOR])
    hedge_formation[stock] = h

optimal_weights = optimize_basket(
    selected_stocks=selected,
    log_prices=log_formation,
    hedge_ratios=hedge_formation,
    allow_short=True,
    verbose=False
)

print('Poids optimaux:')
for stock, w in optimal_weights.items():
    direction = 'LONG ' if w > 0 else 'SHORT'
    print(f'  {direction}  {stock:8s}  w={w:+.4f}   beta={hedge_formation[stock]:+.4f}')

net_beta = sum(hedge_formation[s] * optimal_weights[s] for s in selected)
print(f'\nNet Beta = {net_beta:.6f}  (cible = 0)')
print(f'Sum w = {optimal_weights.sum():.4f}')

## 5. Backtest


In [ ]:
def compute_spread(log_p, weights):
    return (log_p[weights.index] * weights.values).sum(axis=1)


def rolling_zscore(series, window=60):
    mu = series.rolling(window).mean()
    sig = series.rolling(window).std()
    return (series - mu) / sig


def backtest(log_p, prices, weights, z_entry=2.0, z_exit=0.0,
             lookback=60, tx_cost_bps=10, initial_capital=100_000):
    tc = tx_cost_bps / 10_000
    spread = compute_spread(log_p, weights)
    zscore = rolling_zscore(spread, lookback).dropna()

    common = log_p.index.intersection(zscore.index)
    log_p = log_p.loc[common]
    zscore = zscore.loc[common]
    spread = spread.loc[common]

    stocks = weights.index.tolist()
    n = len(stocks)
    w_arr = weights.values
    lp_arr = log_p[stocks].values
    z_arr = zscore.values
    dates = log_p.index

    position = 0
    equity = initial_capital
    entry_lp = None
    entry_z = None
    entry_date = None
    eq_list = []
    pos_list = []
    trade_log = []

    for i in range(len(dates)):
        z = z_arr[i]
        clp = lp_arr[i]

        if position != 0:
            if (position == 1 and z >= z_exit) or (position == -1 and z <= z_exit):
                ret_vec = clp - entry_lp
                gross = float(w_arr.dot(ret_vec)) * position
                cost = tc * n
                net = gross - cost
                equity += net * equity
                trade_log.append({
                    'open_date': entry_date,
                    'close_date': dates[i],
                    'direction': 'LONG' if position == 1 else 'SHORT',
                    'entry_z': round(entry_z, 3),
                    'exit_z': round(z, 3),
                    'gross_ret': round(gross, 5),
                    'cost': round(cost, 5),
                    'net_ret': round(net, 5),
                    'pnl_$': round(net * (equity / (1 + net)), 2)
                })
                position = 0
                entry_lp = None

        if position == 0:
            if z > z_entry:
                position = -1
                entry_lp = clp.copy()
                entry_date = dates[i]
                entry_z = z
                equity    -= tc * n * equity
            elif z < -z_entry:
                position = 1
                entry_lp = clp.copy()
                entry_date = dates[i]
                entry_z = z
                equity    -= tc * n * equity

        eq_list.append(equity)
        pos_list.append(position)

    portfolio = pd.DataFrame({
        'spread': spread.values,
        'zscore': z_arr,
        'position': pos_list,
        'equity': eq_list
    }, index=dates)

    return portfolio, pd.DataFrame(trade_log)

In [ ]:
LOOKBACK = 60
Z_ENTRY = 2.0
Z_EXIT = 0.0
TX_COST_BPS = 10
CAPITAL = 100_000

portfolio, trades = backtest(
    log_p=log_trading, prices=prices_trading, weights=optimal_weights,
    z_entry=Z_ENTRY, z_exit=Z_EXIT, lookback=LOOKBACK,
    tx_cost_bps=TX_COST_BPS, initial_capital=CAPITAL
)

print(f'Nombre de trades: {len(trades)}')
if len(trades) > 0:
    print(f'Gagnants: {(trades["net_ret"] > 0).sum()} / {len(trades)}')
    trades.head(10)

## 6. Metriques


In [ ]:
def compute_metrics(portfolio, trades, risk_free_annual=0.04, label='Strategy'):
    equity = portfolio['equity']
    daily_ret = equity.pct_change().dropna()
    n_years = len(equity) / 252

    total_ret = (equity.iloc[-1] / equity.iloc[0]) - 1
    ann_ret = (1 + total_ret) ** (1 / n_years) - 1
    ann_vol = daily_ret.std() * np.sqrt(252)
    rf_daily = risk_free_annual / 252
    sharpe = (daily_ret.mean() - rf_daily) / daily_ret.std() * np.sqrt(252) if daily_ret.std() > 0 else float('nan')

    rolling_max = equity.cummax()
    drawdown = (equity - rolling_max) / rolling_max
    max_dd = drawdown.min()
    calmar = ann_ret / abs(max_dd) if max_dd != 0 else float('nan')

    if len(trades) > 0:
        win_rate = (trades['net_ret'] > 0).mean()
        avg_win = trades.loc[trades['net_ret'] > 0, 'pnl_$'].mean()
        avg_loss = trades.loc[trades['net_ret'] <= 0, 'pnl_$'].mean()
    else:
        win_rate = avg_win = avg_loss = float('nan')

    return {
        'Strategy': label,
        'Total Return (%)': round(total_ret * 100, 2),
        'Ann. Return (%)': round(ann_ret * 100, 2),
        'Ann. Volatility (%)': round(ann_vol * 100, 2),
        'Sharpe Ratio': round(sharpe, 3),
        'Max Drawdown (%)': round(max_dd * 100, 2),
        'Calmar Ratio': round(calmar, 3),
        '# Trades': len(trades),
        'Win Rate (%)': round(win_rate * 100, 2) if win_rate == win_rate else 'N/A',
        'Avg Win ($)': round(avg_win, 2) if avg_win == avg_win else 'N/A',
        'Avg Loss ($)': round(avg_loss, 2) if avg_loss == avg_loss else 'N/A',
    }


metrics_strat = compute_metrics(portfolio, trades, label='OTT Basket (Gurobi)')
for k, v in metrics_strat.items():
    print(f'  {k:<25s}: {v}')

## 7. Benchmark


In [ ]:
def benchmark_equal_weight(log_p, stocks, initial_capital=100_000, tx_cost_bps=10):
    tc = tx_cost_bps / 10_000
    n = len(stocks)
    log_rets = log_p[stocks].diff().dropna()
    port_ret = (log_rets * (1 / n)).sum(axis=1)
    equity = initial_capital * (1 - tc * n)
    eq_series = equity * np.exp(port_ret.cumsum())
    eq_series = pd.concat([
        pd.Series([equity], index=[log_p.index[0]]),
        eq_series
    ])
    return eq_series


bm_equity = benchmark_equal_weight(
    log_p=log_trading, stocks=selected,
    initial_capital=CAPITAL, tx_cost_bps=TX_COST_BPS
)

bm_ret = bm_equity.pct_change().dropna()
n_years = len(bm_equity) / 252
bm_total = (bm_equity.iloc[-1] / bm_equity.iloc[0]) - 1
bm_ann = (1 + bm_total) ** (1 / n_years) - 1
bm_vol = bm_ret.std() * np.sqrt(252)
bm_sharpe = (bm_ret.mean() - 0.04 / 252) / bm_ret.std() * np.sqrt(252)
bm_dd = ((bm_equity - bm_equity.cummax()) / bm_equity.cummax()).min()
bm_calmar = bm_ann / abs(bm_dd)

metrics_bm = {
    'Strategy': 'Equal-Weight Benchmark',
    'Total Return (%)': round(bm_total * 100, 2),
    'Ann. Return (%)': round(bm_ann * 100, 2),
    'Ann. Volatility (%)': round(bm_vol * 100, 2),
    'Sharpe Ratio': round(bm_sharpe, 3),
    'Max Drawdown (%)': round(bm_dd * 100, 2),
    'Calmar Ratio': round(bm_calmar, 3),
    '# Trades': 1,
}

keys = ['Total Return (%)', 'Ann. Return (%)', 'Ann. Volatility (%)',
        'Sharpe Ratio', 'Max Drawdown (%)', 'Calmar Ratio', '# Trades']

comparison = pd.DataFrame([
    {k: metrics_strat.get(k, 'N/A') for k in keys},
    {k: metrics_bm.get(k, 'N/A') for k in keys}
], index=['OTT Basket (Gurobi)', 'Equal-Weight Benchmark'])

print(comparison.to_string())
comparison

## 8. Performance


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

ax = axes[0]
ax.plot(portfolio.index, portfolio['equity'], label='OTT Basket (Gurobi)', linewidth=1.8)
ax.plot(bm_equity.index, bm_equity.values, label='Equal-Weight Benchmark',
        linewidth=1.5, linestyle='--')
ax.axhline(CAPITAL, color='grey', linestyle=':', linewidth=1)
ax.set_title('Valeur du portfolio (2023-2026)')
ax.set_ylabel('Valeur ($)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())

ax = axes[1]
ax.plot(portfolio.index, portfolio['zscore'], color='steelblue', linewidth=1, label='Z-score')
ax.axhline( Z_ENTRY, color='red',   linestyle='--', linewidth=0.9, label=f'+{Z_ENTRY}sig (entree)')
ax.axhline(-Z_ENTRY, color='green', linestyle='--', linewidth=0.9, label=f'-{Z_ENTRY}sig (entree)')
ax.axhline(0, color='black', linewidth=0.7)
ax.set_title('Z-score du spread (rolling 60j)')
ax.set_ylabel('Z-score')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig('figures/fig_performance.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Sensibilite au seuil Z


In [ ]:
thresholds = [1.5, 1.75, 2.0, 2.25, 2.5]
sensitivity = []

for z in thresholds:
    p, t = backtest(
        log_p=log_trading, prices=prices_trading, weights=optimal_weights,
        z_entry=z, z_exit=Z_EXIT, lookback=LOOKBACK,
        tx_cost_bps=TX_COST_BPS, initial_capital=CAPITAL
    )
    m = compute_metrics(p, t, label=f'z={z}')
    sensitivity.append(m)

sens_df = pd.DataFrame(sensitivity)[[
    'Strategy', 'Ann. Return (%)', 'Sharpe Ratio',
    'Max Drawdown (%)', '# Trades', 'Win Rate (%)'
]].set_index('Strategy')

sens_df